# ==== 【index-tts-vllm-Service】 ====

# clear

In [ ]:
rm -rf ~/.local/share/Trash/*
rm -rf ~/.local/share/Trash/files/*

sudo apt clean


(clean container)
conda clean --all -y
rm -rf /root/.cache/pip
rm -rf /tmp/*

find / -type d -name "__pycache__" -exec rm -rf {} + 2>/dev/null

find / -type f -name "*.pyc" -delete 2>/dev/null

rm -rf /var/log/*
rm -rf /tmp/*

rm -rf /root/miniconda3/envs/index-tts-vllm/lib/python*/site-packages/pip/_vendor/cachecontrol/caches/*
rm -rf /root/miniconda3/pkgs/*

find /root/miniconda3/envs/index-tts-vllm -type d -name "__pycache__" -exec rm -rf {} + 2>/dev/null

find /root/miniconda3/envs/index-tts-vllm -name "*.pyc" -delete 2>/dev/null

// check size 
du -sh /root/miniconda3/envs/index-tts-vllm

# Image history

# Aliyun ACR

# Run Docker containers

## [Test on local MacOS]

Step 1: Build index-tts-vllm Docker image
        change version in: /Users/mac/Documents/GitHub/aliyun_serverless_fc3/my-info/build_docker_image_from_dockerfile.bash

        cd /Users/mac/Documents/GitHub/aliyun_serverless_fc3/my-info && bash ./build_docker_image_from_dockerfile.bash

Step 2: Run index-tts-vllm Docker container
        a. change version in /Users/mac/Documents/GitHub/aliyun_serverless_fc3/my-info/run_index-tts-vllm_docker_from_local_macos_docker_image.bash
        
        b. run this:
        cd /Users/mac/Documents/GitHub/aliyun_serverless_fc3/my-info && bash run_index-tts-vllm_docker_from_local_macos_docker_image.bash


## [For WQ docker image]

# SpeakerID


In [ ]:
d10bb92ab03c9e8c301a3ab2ca11fba3 : jay_promptvn

(wm speakerId)
d10bb92ab03c9e8c301a3ab2ca11fba3  [with nosise]

7678676fff6d4086132cf6c6a575f571 [no noise]
4582b310374bcb6633f2d459bd29811c

# Other

# >>model
//>>download model
cd /mnt/index-tts-vllm
modelscope download --model kusuriuri/Index-TTS-1.5-vLLM --local_dir ./checkpoints/Index-TTS-1.5-vLLM

In [ ]:
# ===== how to run index-tts-vllm at server from scratch =====
# Step 1：(upload local files to server using Termius)
# upload these files:
# from local:
# /Users/mac/Documents/GitHub/index-tts-vllm/assets/
# to server:
# /mnt/index-tts-vllm/assets/

sudo apt update && sudo apt install -y python3-pip
sudo ln -s /usr/bin/python3 /usr/bin/python

# Step 2：(run container in background and keep terminal at host)
# 这个命令会启动容器在后台运行，终端保持在宿主机
version="1.4.9" && server_name="index-tts-vllm" && image_prefix="d.watchfun.cn/jims57" && image_name="${image_prefix}/${server_name}" && tag="v${version}" && docker rm "${server_name}" -f && docker run --gpus all -d -p 9001:9001 --shm-size=4g -w /mnt/index-tts-vllm -v /mnt/index-tts-vllm/checkpoints:/mnt/index-tts-vllm/checkpoints -v /mnt/index-tts-vllm/assets:/mnt/index-tts-vllm/assets -v /mnt/index-tts-vllm/logs:/mnt/index-tts-vllm/logs -v /mnt/index-tts-vllm/savedAudioFiles:/mnt/index-tts-vllm/savedAudioFiles -e API_PORT=9001 --name "${server_name}" "${image_name}:${tag}" tail -f /dev/null

# Step 3：(在容器内后台启动api_server.py)
# 在后台运行api_server.py并保存日志到文件
docker exec -d index-tts-vllm /bin/bash -c "cd /mnt/index-tts-vllm && nohup /root/miniconda3/envs/index-tts-vllm/bin/python api_server.py --port 6006 --model_dir /mnt/index-tts-vllm/checkpoints/Index-TTS-1.5-vLLM --gpu_memory_utilization 0.25 > /mnt/index-tts-vllm/logs/api_server_py_$(date +%Y%m%d_%H%M%S).log 2>&1 &"

# Step 4：(查看api_server.py日志，确认启动成功)
# 等待约30秒，然后查看日志确认api_server.py启动成功
sleep 30 && docker exec -it index-tts-vllm /bin/bash -c "tail -f /mnt/index-tts-vllm/logs/api_server_py_*.log"

# Step 5：(在容器内后台运行api.py)
# 确认api_server.py启动成功后，按Ctrl+C停止查看日志，然后运行以下命令
docker exec -d index-tts-vllm /bin/bash -c "cd /mnt/index-tts-vllm && nohup /root/miniconda3/envs/index-tts-vllm/bin/python api.py --port 9001 > /mnt/index-tts-vllm/logs/api_py_$(date +%Y%m%d_%H%M%S).log 2>&1 &"

# Step 6：(查看api.py日志，确认启动成功)
# 等待几秒钟，然后查看日志确认api.py启动成功
sleep 5 && docker exec -it index-tts-vllm /bin/bash -c "tail -f /mnt/index-tts-vllm/logs/api_py_*.log"

# ===== 常用命令 =====
# 查看容器状态
# docker ps | grep index-tts-vllm

# 查看api_server.py和api.py进程
# docker exec index-tts-vllm /bin/bash -c "ps aux | grep python"

# 查看api_server.py日志
# docker exec -it index-tts-vllm /bin/bash -c "tail -f /mnt/index-tts-vllm/logs/api_server_py_*.log"

# 查看api.py日志
# docker exec -it index-tts-vllm /bin/bash -c "tail -f /mnt/index-tts-vllm/logs/api_py_*.log"

# 进入容器交互式终端
# docker exec -it index-tts-vllm /bin/bash

# ===== 停止服务命令 =====
# 查找api_server.py进程ID
# docker exec index-tts-vllm /bin/bash -c "ps aux | grep api_server.py | grep -v grep | awk '{print \$2}'"

# 停止api_server.py进程
# docker exec index-tts-vllm /bin/bash -c "ps aux | grep api_server.py | grep -v grep | awk '{print \$2}' | xargs -r kill -9"

# 查找api.py进程ID
# docker exec index-tts-vllm /bin/bash -c "ps aux | grep api.py | grep -v grep | awk '{print \$2}'"

# 停止api.py进程
# docker exec index-tts-vllm /bin/bash -c "ps aux | grep api.py | grep -v grep | awk '{print \$2}' | xargs -r kill -9"

# 一次性停止所有Python进程
# docker exec index-tts-vllm /bin/bash -c "pkill -9 python"

# 停止并删除容器
# docker stop index-tts-vllm && docker rm index-tts-vllm